# SEED baseline: evaluación 0, fold 0

Este notebook es una interfaz de ejecución y visualización. La lógica reproducible vive en los scripts `.py`: no se redefine SEED, el preprocesamiento ni las métricas legacy aquí.

Ejecuta las celdas en orden. Las celdas que lanzan cómputo están protegidas por interruptores `False` para evitar entrenar o evaluar por accidente.

In [ ]:
import os

os.environ["EEG_DATA_ROOT"] = r"C:\Users\ferna\Downloads\data"

## 1. Entorno y rutas privadas

Antes de abrir Jupyter, configura la variable de entorno en la terminal que lo inicia. No guardes rutas privadas en el notebook:

```powershell
$env:EEG_DATA_ROOT = 'C:\ruta\a\eeg_private_data'
.\.venv\Scripts\jupyter.exe lab
```

En la instalación local actual los HDF5 pueden estar dentro de un subdirectorio `data/`, mientras los pesos están directamente bajo `EEG_DATA_ROOT`; las celdas siguientes soportan ambos casos sin copiar archivos.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'train.py').is_file() and (REPO_ROOT.parent / 'train.py').is_file():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'train.py').is_file():
    raise RuntimeError('Abre Jupyter desde la raíz del repositorio.')

root_value = os.environ.get('EEG_DATA_ROOT')
if not root_value:
    raise RuntimeError('Define EEG_DATA_ROOT en la terminal antes de iniciar Jupyter.')

DATA_ROOT = Path(root_value).expanduser().resolve()
H5_ROOT = DATA_ROOT / 'data' if (DATA_ROOT / 'data' / 'moda_augmented_s5.h5').is_file() else DATA_ROOT
AUGMENTED_H5 = H5_ROOT / 'moda_augmented_s5.h5'
RAW_H5 = H5_ROOT / 'moda_moda.h5'
LEGACY_WEIGHTS = DATA_ROOT / 'model_weights_best_original'

for expected in (AUGMENTED_H5, RAW_H5, LEGACY_WEIGHTS):
    if not expected.exists():
        raise FileNotFoundError(f'Ruta privada faltante: {expected}')

print(f'Python del kernel: {sys.executable}')
print(f'EEG_DATA_ROOT: {DATA_ROOT}')
print(f'HDF5: {H5_ROOT}')
print(f'Pesos legacy: {LEGACY_WEIGHTS}')

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {properties.total_memory / 1024**3:.2f} GiB')

In [ ]:
def run_script(*arguments: str) -> None:
    """Run a repository script with the current Jupyter kernel."""
    command = [sys.executable, *map(str, arguments)]
    print('>', ' '.join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)

FOLD_ARGUMENTS = ('--evaluation', '0', '--fold', '0')
BASELINE_NPZ = REPO_ROOT / 'outputs' / 'baseline' / 'predictions_legacy_eval_1_fold_1.npz'
TRAIN_OUTPUT = REPO_ROOT / 'outputs' / 'full_fold0' / 'train_eval_0_fold_0_seed_42'
print('Scripts se ejecutarán desde:', REPO_ROOT)

## 2. Verificación ligera: tests y EDA

Activa estas ejecuciones sólo si necesitas repetirlas. Los resultados existentes se pueden inspeccionar más abajo.

In [ ]:
RUN_LIGHT_CHECKS = False

if RUN_LIGHT_CHECKS:
    run_script('-m', 'pytest', '-q')
    run_script('scripts/eda.py', '--data-root', str(H5_ROOT))
else:
    print('Cambia RUN_LIGHT_CHECKS a True para repetir tests y EDA.')

In [ ]:
eda_path = REPO_ROOT / 'outputs' / 'eda' / 'eda_summary.json'
if eda_path.exists():
    eda = json.loads(eda_path.read_text())
    audit = eda['fold_audit']
    print('Sujetos:', eda['augmented']['subjects'])
    print('Registros:', eda['augmented']['records'])
    print('Fracción temporal positiva:', f"{eda['augmented']['positive_temporal_fraction']:.2%}")
    print('Sujetos en folds:', audit['subjects_in_folds'])
    print('Leakage máximo:', max(max(f['train_val_overlap'], f['train_test_overlap'], f['val_test_overlap']) for f in audit['folds']))
else:
    print('Aún no existe eda_summary.json. Ejecuta la celda anterior con RUN_LIGHT_CHECKS=True.')

## 3. Entrenamiento debug

El debug usa sólo 128 ventanas y dos épocas. Valida GPU, shapes, checkpoints y exportación; **no** es una estimación de rendimiento del fold. El script fuerza `num_workers=0` en este modo.

In [ ]:
RUN_DEBUG_TRAINING = False
BATCH_SIZE = 4  # Probar 8 sólo después de confirmar que 4 no genera CUDA OOM.

if RUN_DEBUG_TRAINING:
    run_script(
        'train.py', *FOLD_ARGUMENTS, '--debug',
        '--data-root', str(H5_ROOT), '--device', 'cuda',
        '--batch-size', str(BATCH_SIZE), '--num-workers', '2',
    )
else:
    print('Cambia RUN_DEBUG_TRAINING a True para lanzar el debug.')

In [ ]:
metrics_path = TRAIN_OUTPUT / 'metrics.json'
if metrics_path.exists():
    training = json.loads(metrics_path.read_text())
    history = training['history']
    epochs = [item['epoch'] for item in history]
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, [item['train_loss'] for item in history], marker='o', label='train')
    plt.plot(epochs, [item['val_loss'] for item in history], marker='o', label='validation')
    plt.xlabel('Época'); plt.ylabel('BCE loss'); plt.title('SEED: loss temporal')
    plt.legend(); plt.grid(alpha=.25); plt.show()
    print('Dispositivo:', training['device'])
    print('Mejor loss:', training['best_val_loss'])
    print('Modo overfit:', training['overfit_mode_uses_training_subset_for_evaluation'])
else:
    print('Aún no existe metrics.json para el debug.')

## 4. Baseline legacy y uncertainty post-hoc

La evaluación usa **exclusivamente** `model_weights_best_original`. La incertidumbre se calcula desde `sigmoid(logits)` y no altera el postprocesamiento, pairing ni IoU legacy.

In [ ]:
RUN_BASELINE = False

if RUN_BASELINE:
    run_script(
        'evaluate_baseline.py', *FOLD_ARGUMENTS,
        '--data-root', str(DATA_ROOT),
        '--data-path', str(RAW_H5),
        '--weights-dir', str(LEGACY_WEIGHTS),
        '--device', 'cuda',
    )
else:
    print('Cambia RUN_BASELINE a True para reproducir el baseline legacy.')

In [ ]:
RUN_UNCERTAINTY = False

if RUN_UNCERTAINTY:
    run_script('analyze_uncertainty.py', '--predictions', str(BASELINE_NPZ))
else:
    print('Cambia RUN_UNCERTAINTY a True para calcular confidence, uncertainty y entropy.')

In [ ]:
summary_path = BASELINE_NPZ.parent / 'uncertainty_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
else:
    print('Aún no existe uncertainty_summary.json.')

In [ ]:
uncertainty_path = BASELINE_NPZ.parent / 'predictions_with_uncertainty.npz'
if uncertainty_path.exists():
    arrays = np.load(uncertainty_path)
    required = {'y_true', 'probabilities', 'processed_pred', 'entropy', 'uncertainty'}
    missing = required - set(arrays.files)
    if missing:
        raise KeyError(f'Faltan arrays: {sorted(missing)}')

    sample_index, start, length = 0, 0, 4000
    stop = min(start + length, arrays['probabilities'].shape[1])
    time_seconds = np.arange(start, stop) / 200.0
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(time_seconds, arrays['probabilities'][sample_index, start:stop], lw=1, label='P(spindle)')
    axes[0].plot(time_seconds, arrays['y_true'][sample_index, start:stop], alpha=.6, label='Etiqueta')
    axes[0].plot(time_seconds, arrays['processed_pred'][sample_index, start:stop], alpha=.6, label='Predicción procesada')
    axes[0].set_ylabel('Probabilidad / máscara'); axes[0].legend(loc='upper right')
    axes[1].plot(time_seconds, arrays['entropy'][sample_index, start:stop], color='tab:orange', lw=1)
    axes[1].set_ylabel('Entropía binaria')
    axes[2].plot(time_seconds, arrays['uncertainty'][sample_index, start:stop], color='tab:red', lw=1)
    axes[2].set_ylabel('1 - confianza'); axes[2].set_xlabel('Tiempo (s)')
    fig.suptitle('Fold 0: probabilidad, etiqueta y uncertainty temporal')
    plt.tight_layout(); plt.show()

    error = arrays['y_pred'] != arrays['y_true']
    entropy = arrays['entropy']
    rng = np.random.default_rng(42)
    correct_values = entropy[~error].ravel()
    incorrect_values = entropy[error].ravel()
    correct_values = rng.choice(correct_values, size=min(200_000, correct_values.size), replace=False)
    incorrect_values = rng.choice(incorrect_values, size=min(200_000, incorrect_values.size), replace=False)
    plt.figure(figsize=(8, 4))
    plt.hist(correct_values, bins=50, density=True, alpha=.6, label='Correcto')
    plt.hist(incorrect_values, bins=50, density=True, alpha=.6, label='Incorrecto')
    plt.xlabel('Entropía binaria'); plt.ylabel('Densidad'); plt.title('Uncertainty por acierto temporal')
    plt.legend(); plt.grid(alpha=.2); plt.show()
else:
    print('Aún no existe predictions_with_uncertainty.npz.')

## Próximos pasos

Cuando el fold 0 normal esté aprobado, mantén el entrenamiento en los scripts `.py` y crea otro notebook de resumen para los cinco folds intermedios y, después, los quince folds finales. No cambies umbrales, IoU, pairing ni postprocesamiento legacy desde una celda de notebook.

## 6. Resumen reproducible para la entrega

Esta seccion resume los artefactos ya generados. No recalcula el detector ni modifica sus umbrales: solo lee `baseline_metrics.json` y `uncertainty_summary.json`.

In [ ]:
from IPython.display import Markdown, display

metrics_path = BASELINE_NPZ.parent / 'baseline_metrics.json'
uncertainty_summary_path = BASELINE_NPZ.parent / 'uncertainty_summary.json'

if not metrics_path.exists() or not uncertainty_summary_path.exists():
    missing = [path.name for path in (metrics_path, uncertainty_summary_path) if not path.exists()]
    raise FileNotFoundError(f'Faltan artefactos para el resumen: {missing}')

baseline_report = json.loads(metrics_path.read_text())
uq_report = json.loads(uncertainty_summary_path.read_text())

report_lines = [
    '### Resultado del baseline (un fold de test)',
    '',
    '| F1 a IoU 0.2 | Precision | Recall | Eventos evaluados |',
    '| ---: | ---: | ---: | ---: |',
    '| {:.2f}% | {:.2f}% | {:.2f}% | {:,} |'.format(baseline_report['f1_score'], baseline_report['precision'], baseline_report['recall'], baseline_report['total_results']),
    '',
    '### Incertidumbre temporal post-hoc',
    '',
    '| Medida | Timesteps correctos | Timesteps incorrectos |',
    '| --- | ---: | ---: |',
    '| Entropia media | {:.4f} | {:.4f} |'.format(uq_report['correct_entropy']['mean'], uq_report['incorrect_entropy']['mean']),
    '| Incertidumbre media | {:.4f} | {:.4f} |'.format(uq_report['correct_uncertainty']['mean'], uq_report['incorrect_uncertainty']['mean']),
    '',
    '**AUROC de incertidumbre para detectar error temporal:** {:.4f}'.format(uq_report['error_detection_auroc_uncertainty']),
]
display(Markdown(chr(10).join(report_lines)))


### Interpretacion

El baseline mide deteccion de eventos tras el postprocesamiento legacy: F1, precision y recall se calculan con IoU 0.2. En cambio, la incertidumbre de este MVP se calcula por timestep a partir de las probabilidades del mismo detector. Si la entropia y la incertidumbre son mayores en los errores, y el AUROC es mayor que 0.5, el score ordena los errores mejor que el azar.

Este resultado es prometedor si se observa esa separacion, pero su alcance es limitado: corresponde a un solo fold, a incertidumbre temporal determinista y no demuestra aun incertidumbre epistemica ni calidad de incertidumbre a nivel de evento. Por lo tanto, no debe presentarse como resultado clinico ni como rendimiento final del modelo.

### Checklist de esta entrega

- [x] Particion por sujeto y fold identificados (`evaluation 0`, `fold 0`, equivalentes a `eval_1/fold_1`).
- [x] Baseline evaluado sobre sujetos de test con protocolo legacy, IoU 0.2 y pesos legacy.
- [x] Entropia/confianza calculadas sin cambiar el detector.
- [x] Comparacion entre aciertos y errores, con AUROC de deteccion de errores temporales.
- [ ] Repetir el protocolo en los 15 pares evaluacion/fold y reportar dispersion entre folds.
- [ ] Comparar una tecnica de incertidumbre nueva bajo el mismo protocolo.


## Siguiente fase experimental

Repetir baseline e incertidumbre en los 15 pares antes de concluir que el resultado se generaliza. Mantener el entrenamiento en los scripts `.py`; no cambiar umbrales, IoU, pairing ni postprocesamiento legacy desde una celda de notebook.